# Vedic Transformer Training with BPE (Google Colab)

This notebook trains a Byte Pair Encoding (BPE) Transformer for Vedic Sanskrit.

## Setup Instructions
1.  **Select GPU**: Go to `Runtime` -> `Change runtime type` -> `T4 GPU` (or better).
2.  **Upload Data to Drive**: Ensure the following are in your `vedAI` folder on Google Drive:
    - `veda_train_accented.txt` (the processed corpus)
    - `bpe_tokenizer/vocab.json`
    - `bpe_tokenizer/merges.txt` 


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Project path in Google Drive
PROJECT_PATH = '/content/drive/MyDrive/vedAI'
import os
if not os.path.exists(PROJECT_PATH):
    print(f'Please ensure the folder exists at {PROJECT_PATH} or update this path.')

In [ ]:
!pip install tokenizers

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from tokenizers import ByteLevelBPETokenizer
import json
import os
import sys
import math

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# --- Configuration & Hyperparameters (Optimized for 5MB Corpus) ---
DATASET_TYPE = 'accented' 
BATCH_SIZE = 64
BLOCK_SIZE = 512 # Increased for better metrical context
MAX_ITERS = 10000 # Reduced for smaller dataset
EVAL_INTERVAL = 500
LEARNING_RATE = 3e-4
MIN_LR = 3e-5
WARMUP_ITERS = 500
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_ITERS = 50

# Model size (Approx 5.8M parameters - Ideal for 5MB corpus)
N_EMBD = 256
N_HEAD = 8 # (256 / 8 = 32 Head size)
N_LAYER = 6
DROPOUT = 0.2
WEIGHT_DECAY = 0.1
# -----------------------------------------------------

In [ ]:
def load_bpe_data():
    text_path = os.path.join(PROJECT_PATH, f'veda_train_{DATASET_TYPE}.txt')
    tokenizer_dir = os.path.join(PROJECT_PATH, 'bpe_tokenizer')

    if not os.path.exists(text_path):
        raise FileNotFoundError(f"\u274c Error: Could not find {text_path}")
    if not os.path.exists(tokenizer_dir):
        raise FileNotFoundError(f"\u274c Error: Could not find tokenizer at {tokenizer_dir}")

    print('Loading Tokenizer...')
    tokenizer = ByteLevelBPETokenizer(
        os.path.join(tokenizer_dir, "vocab.json"),
        os.path.join(tokenizer_dir, "merges.txt")
    )
    vocab_size = tokenizer.get_vocab_size()
    print(f"BPE Vocab Size: {vocab_size}")

    print('Loading dataset...')
    with open(text_path, 'r', encoding='utf-8') as f:
        text = f.read()

    print('Encoding text...')
    ids = tokenizer.encode(text).ids
    data_tensor = torch.tensor(ids, dtype=torch.long)
    print(f'Dataset size: {len(data_tensor)} tokens')

    n = int(0.9 * len(data_tensor))
    train_data = data_tensor[:n]
    val_data = data_tensor[n:]
    
    return tokenizer, vocab_size, train_data, val_data

tokenizer, vocab_size, train_data, val_data = load_bpe_data()

In [ ]:
def get_lr(it):
    if it < WARMUP_ITERS:
        return LEARNING_RATE * it / WARMUP_ITERS
    if it > MAX_ITERS:
        return MIN_LR
    decay_ratio = (it - WARMUP_ITERS) / (MAX_ITERS - WARMUP_ITERS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (LEARNING_RATE - MIN_LR)

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data_split[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data_split[i+1:i+BLOCK_SIZE+1] for i in ix])
    x, y = x.to(DEVICE), y.to(DEVICE)
    return x, y

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(DROPOUT),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class VedLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks = nn.Sequential(*[Block(N_EMBD, n_head=N_HEAD) for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=DEVICE))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50, eos_token=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if eos_token is not None and idx_next.item() == eos_token: break
        return idx

In [ ]:
model = VedLanguageModel().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

print(f'Model Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M')
print(f'Training on: {DEVICE}')

eos_token = tokenizer.token_to_id("<eos>")

for iter in range(MAX_ITERS):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups: param_group['lr'] = lr

    if iter % EVAL_INTERVAL == 0 or iter == MAX_ITERS - 1:
        losses = estimate_loss(model)
        print(f'step {iter}: lr {lr:.6f}, train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}')

        # Start with <RIG> to see if style-conditioning is working
        context = torch.tensor([[tokenizer.token_to_id("<RIG>")]], dtype=torch.long, device=DEVICE)
        gen_tokens = model.generate(context, max_new_tokens=200, eos_token=eos_token)[0].tolist()
        print(f'--- Sample ---\n{tokenizer.decode(gen_tokens).replace("\n", " / ")}\n--------------')

    xb, yb = get_batch('train')
    with torch.amp.autocast(device_type=DEVICE if DEVICE == 'cuda' else 'cpu', enabled=(DEVICE == 'cuda')):
        logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    if DEVICE == 'cuda':
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

print('Training Complete!')

In [ ]:
import shutil
# Define export path
export_path = os.path.join(PROJECT_PATH, 'final_bpe_model_export')
os.makedirs(export_path, exist_ok=True)

# Save the final model weights
torch.save(model.state_dict(), os.path.join(export_path, 'model_weights.pt'))

print(f'Done! Model exported to: {export_path}')
print('Ensure you also keep the bpe_tokenizer folder to decode the generated text later!')